In [1]:
# %%
"""
MINI CHUNK 0 DETERMINISM TRIPWIRE (RUN 1 VS RUN 2)
Verifies bit-level identity (SHA-256 match and atol = 0.0) across two independent
training instantiations of Chunk 0 on identical seeds.
"""

import copy
import gc
import hashlib
import os
from pathlib import Path
import random
import sys
from typing import Any, Dict, Tuple

import numpy as np
import pandas as pd
import torch

# Ensure local repository root is resolved
current_dir = Path.cwd()
project_root = current_dir if (current_dir / "core").exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from core.paths import LOCAL_DATA_DIR
from core.settings import CacheConfig, TradingConfig
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from rl_discovery.adapter import (
    ObservationScaler,
    RLVRGymEnv,
    make_eval_env,
    make_stratified_train_envs,
)
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.trainer import PPOTrainer, RolloutBuffer
from rl_discovery.validator import AgentEvaluator


def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def compute_agent_sha256(
    agent: AbsoluteZeroAgent, scaler: ObservationScaler
) -> Tuple[str, Dict[str, str]]:
    hasher = hashlib.sha256()
    tensor_hashes = {}
    for name, param in sorted(agent.state_dict().items()):
        arr = param.detach().cpu().numpy()
        h = hashlib.sha256(arr.tobytes()).hexdigest()
        tensor_hashes[name] = h
        hasher.update(name.encode("utf-8"))
        hasher.update(arr.tobytes())

    scaler_mean_h = hashlib.sha256(scaler.mean.tobytes()).hexdigest()
    scaler_var_h = hashlib.sha256(scaler.var.tobytes()).hexdigest()
    tensor_hashes["scaler.mean"] = scaler_mean_h
    tensor_hashes["scaler.var"] = scaler_var_h
    hasher.update(scaler.mean.tobytes())
    hasher.update(scaler.var.tobytes())

    return hasher.hexdigest(), tensor_hashes


def execute_mini_chunk_run(
    seed: int,
    epochs: int = 2,
    num_envs: int = 4,
    num_steps: int = 64,
    mini_batch_size: int = 64,
    device: torch.device = torch.device("cpu"),
) -> Tuple[AbsoluteZeroAgent, ObservationScaler, str, Dict[str, str], float]:
    set_seed(seed)
    config = TradingConfig()

    data = load_processed_data()
    df_ohlcv = data.df_ohlcv
    macro_df = data.macro_df
    df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
    master_cal = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

    cache_file = LOCAL_DATA_DIR / CacheConfig.get_filename()
    feature_cube = pd.read_parquet(cache_file)
    valid_dates = set(feature_cube.index.get_level_values("Date").unique())
    trading_calendar = master_cal[master_cal.isin(valid_dates)]

    simple_ret_matrix = df_close.pct_change(1, fill_method=None).shift(-1)
    simple_ret_matrix["CASH"] = 0.0

    # Chunk 0 Training Bounds
    t_start = pd.Timestamp("2016-01-04")
    t_end = pd.Timestamp("2018-01-04")  # Compact 2-year window for rapid tripwire
    cal_train = trading_calendar[
        (trading_calendar >= t_start) & (trading_calendar <= t_end)
    ]
    cal_train = cal_train[: -config.holding_period]

    obs_dim = 3 * feature_cube.shape[1] + len(macro_df.columns)
    action_dim = feature_cube.shape[1] + 4

    agent = AbsoluteZeroAgent(obs_dim=obs_dim, action_dim=action_dim).to(device)

    envs = make_stratified_train_envs(
        feature_cube=feature_cube,
        simple_ret_matrix=simple_ret_matrix,
        calendar=cal_train,
        macro_df=macro_df,
        config=config,
        num_envs=num_envs,
        episode_steps=num_steps,
        hist_cutoff_idx=None,
        initial_scaler_state=None,
        seed=seed,
    )

    trainer = PPOTrainer(
        agent=agent,
        lr=2.0e-4,
        critic_lr=8.0e-4,
        clip_coef=config.clip_coef,
        clip_vloss=True,
        ent_coef=config.entropy_coef_start,
        seed=seed,
    )

    buffer = RolloutBuffer(
        num_steps=num_steps,
        num_envs=num_envs,
        obs_dim=obs_dim,
        action_dim=action_dim,
        device=device,
        gamma=config.gamma,
        gae_lambda=config.gae_lambda,
    )

    total_observed_reward = 0.0

    for ep in range(epochs):
        obs, _ = envs.reset()
        for step in range(num_steps):
            obs_tensor = torch.tensor(obs, dtype=torch.float32, device=device)
            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(obs_tensor)

            next_obs, rewards, terminations, truncations, _ = envs.step(
                action.cpu().numpy()
            )
            buffer.add(
                obs=obs,
                action=action,
                logprob=logprob,
                reward=rewards * 25.0,
                value=value,
                terminations=terminations,
                truncations=truncations,
            )
            total_observed_reward += float(np.sum(rewards))
            obs = next_obs

        obs_tensor = torch.tensor(obs, dtype=torch.float32, device=device)
        next_term_tensor = torch.tensor(
            terminations, dtype=torch.float32, device=device
        )
        with torch.no_grad():
            next_values = agent.get_value(obs_tensor)
        buffer.compute_advantages(next_values, next_termination=next_term_tensor)

        trainer.update(
            buffer=buffer,
            update_epochs=2,
            mini_batch_size=mini_batch_size,
            stratified_sampling=False,
        )
        buffer.step = 0

    active_scaler = envs.get_attr("scaler")[0]
    envs.close()

    overall_hash, tensor_hashes = compute_agent_sha256(agent, active_scaler)
    return agent, active_scaler, overall_hash, tensor_hashes, total_observed_reward


# =============================================================================
# EXECUTE REPRODUCIBILITY EXPERIMENT (RUN 1 VS RUN 2)
# =============================================================================
target_seed = 42
exec_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print(
    f"🚀 LAUNCHING E2E DETERMINISM TRIPWIRE [Seed: {target_seed} | Device: {exec_device}]"
)
print("=" * 80)

print("\n▶️ Executing Run 1...")
agent_1, scaler_1, hash_1, t_hashes_1, rew_1 = execute_mini_chunk_run(
    seed=target_seed, device=exec_device
)
print(f"   Run 1 Digest: {hash_1} | Total Rollout Reward: {rew_1:.6f}")

# Aggressive cache purge to eliminate memory artifacts
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n▶️ Executing Run 2 (Fresh Instantiation)...")
agent_2, scaler_2, hash_2, t_hashes_2, rew_2 = execute_mini_chunk_run(
    seed=target_seed, device=exec_device
)
print(f"   Run 2 Digest: {hash_2} | Total Rollout Reward: {rew_2:.6f}")

# =============================================================================
# MATHEMATICAL DISCREPANCY AUDIT
# =============================================================================
print("\n" + "=" * 80)
print("🔍 MATHEMATICAL DISCREPANCY & LAYER-BY-LAYER BIT-LEVEL AUDIT")
print("=" * 80)

max_weight_diff = 0.0
divergent_layers = []

for name, param1 in agent_1.state_dict().items():
    param2 = agent_2.state_dict()[name]
    diff = torch.max(torch.abs(param1 - param2)).item()
    h1 = t_hashes_1[name]
    h2 = t_hashes_2[name]
    status = "✅ MATCH" if h1 == h2 and diff == 0.0 else "❌ DIVERGED"
    if diff > 0.0 or h1 != h2:
        divergent_layers.append((name, diff))
    max_weight_diff = max(max_weight_diff, diff)
    print(f"  • {name:<35} | Diff: {diff:.2e} | SHA256: {h1[:10]}... | {status}")

# Scaler verification
scaler_mean_diff = float(np.max(np.abs(scaler_1.mean - scaler_2.mean)))
scaler_var_diff = float(np.max(np.abs(scaler_1.var - scaler_2.var)))
print(
    f"  • {'scaler.mean':<35} | Diff: {scaler_mean_diff:.2e} | {'✅ MATCH' if scaler_mean_diff == 0.0 else '❌ DIVERGED'}"
)
print(
    f"  • {'scaler.var':<35} | Diff: {scaler_var_diff:.2e} | {'✅ MATCH' if scaler_var_diff == 0.0 else '❌ DIVERGED'}"
)

print("-" * 80)
print(f"Reward Divergence (|Rew_1 - Rew_2|) : {abs(rew_1 - rew_2):.2e}")
print(f"Max Weight Tensor Divergence       : {max_weight_diff:.2e}")
print(f"Master Checkpoint SHA-256 Run 1     : {hash_1}")
print(f"Master Checkpoint SHA-256 Run 2     : {hash_2}")

# Institutional Gate
assert (
    hash_1 == hash_2
), f"🚨 CRITICAL DETERMINISM BREACH! Checkpoint hashes diverge: {hash_1} != {hash_2}"
assert (
    max_weight_diff == 0.0
), f"🚨 Parameter weights drifted! Max abs error: {max_weight_diff}"
assert (
    scaler_mean_diff == 0.0 and scaler_var_diff == 0.0
), "🚨 Observation Scaler drifted across runs!"

print("\n" + "#" * 80)
print(
    "🏆 VERDICT: 100% BIT-LEVEL DETERMINISTIC IDENTITY CONFIRMED (IEEE 754 float32 atol = 0.0)"
)
print("#" * 80)
# %%

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output
TRADES_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\trades

🚀 LAUNCHING E2E DETERMINISM TRIPWIRE [Seed: 42 | Device: cpu]

▶️ Executing Run 1...
   Run 1 Digest: 201eee240373ffc56337d770356e95022152c0d2b73d323d4b513ec16b38a66b | Total Rollout Reward: 0.018992

▶️ Executing Run 2 (Fresh Instantiation)...
   Run 2 Digest: 201eee240373ffc56337d770356e95022152c0d2b73d323d4b513ec16b38a66b | Total Rollout Reward: 0.018992

🔍 MATHEMATICAL DISCREPANCY & LAYER-BY-LAYER BIT-LEVEL AUDIT
  • actor_logstd               

In [2]:
# %%
"""
Manifest Deep Diff Probe (Dynamic Generation Comparison).
Recursively audits nested dicts and lists across trading_config, chunk_specs,
data_lineage, environment, and checkpoint champion training epochs between any two generations.
"""

import argparse
import json
from pathlib import Path
from typing import Any, Dict, Tuple
import torch


def load_manifest(gen: int) -> Dict[str, Any]:
    paths = [
        Path(f"output/canonical_anchors/run_metadata_gen{gen}.json"),
        Path(f"output/model_checkpoints/walk_forward_gen{gen}/run_metadata.json"),
        Path(f"output/run_metadata_gen{gen}.json"),
    ]
    for p in paths:
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError(f"Could not find manifest for Gen {gen}")


def deep_diff(
    v1: Any,
    v2: Any,
    prefix: str = "",
    label1: str = "GenA",
    label2: str = "GenB",
) -> Dict[str, Dict[str, Any]]:
    """Recursively diffs arbitrary combinations of dicts, lists, and primitives."""
    diffs = {}

    if isinstance(v1, dict) and isinstance(v2, dict):
        all_keys = sorted(set(v1.keys()).union(set(v2.keys())))
        for k in all_keys:
            path = f"{prefix}.{k}" if prefix else str(k)
            diffs.update(
                deep_diff(
                    v1.get(k), v2.get(k), prefix=path, label1=label1, label2=label2
                )
            )

    elif isinstance(v1, list) and isinstance(v2, list):
        if len(v1) != len(v2):
            diffs[f"{prefix}.__len__"] = {label1: len(v1), label2: len(v2)}
        max_len = max(len(v1), len(v2))
        for i in range(max_len):
            val1 = v1[i] if i < len(v1) else "<MISSING>"
            val2 = v2[i] if i < len(v2) else "<MISSING>"
            path = f"{prefix}[{i}]"
            diffs.update(
                deep_diff(val1, val2, prefix=path, label1=label1, label2=label2)
            )

    else:
        if v1 != v2:
            diffs[prefix] = {label1: v1, label2: v2}

    return diffs


def get_target_generations() -> Tuple[int, int]:
    """Retrieves generation IDs from CLI arguments or prompts the user interactively."""
    parser = argparse.ArgumentParser(
        description="Deep diff manifest probe between two generations."
    )
    parser.add_argument(
        "gen1", type=int, nargs="?", help="First generation ID (e.g., 23)"
    )
    parser.add_argument(
        "gen2", type=int, nargs="?", help="Second generation ID (e.g., 24)"
    )

    # parse_known_args prevents notebook/interactive kernel flags (e.g. -f) from throwing errors
    args, _ = parser.parse_known_args()

    gen1 = args.gen1
    gen2 = args.gen2

    if gen1 is None:
        while True:
            try:
                gen1 = int(input("Enter base generation ID (e.g., 23): ").strip())
                break
            except ValueError:
                print("❌ Invalid input. Please enter an integer.")

    if gen2 is None:
        while True:
            try:
                gen2 = int(input("Enter comparison generation ID (e.g., 24): ").strip())
                break
            except ValueError:
                print("❌ Invalid input. Please enter an integer.")

    return gen1, gen2


def main():
    gen1, gen2 = get_target_generations()
    label1 = f"Gen{gen1}"
    label2 = f"Gen{gen2}"

    print("=" * 70)
    print(f"🔍 DEEP DIFF: GENERATION {gen1} vs GENERATION {gen2} CONFIGURATIONS")
    print("=" * 70)

    try:
        m1 = load_manifest(gen1)
        m2 = load_manifest(gen2)
    except FileNotFoundError as e:
        print(f"❌ Error loading manifests: {e}")
        return

    sections = ["trading_config", "chunk_specs", "data_lineage", "environment"]

    for section in sections:
        c1 = m1.get(section, {})
        c2 = m2.get(section, {})
        diffs = deep_diff(c1, c2, prefix=section, label1=label1, label2=label2)

        # Ignore non-functional noise
        meaningful_diffs = {
            k: v
            for k, v in diffs.items()
            if not any(
                skip in k.lower()
                for skip in ["timestamp", "mtime", "manifest_path", "cache_file_mtime"]
            )
        }

        if meaningful_diffs:
            print(f"\n⚡ Differences detected in [{section}]:")
            for k, v in meaningful_diffs.items():
                print(f"  • {k}:")
                print(f"      Gen {gen1} -> {v[label1]}")
                print(f"      Gen {gen2} -> {v[label2]}")
        else:
            print(
                f"✅ [{section}] is 100% bitwise IDENTICAL between Gen {gen1} and Gen {gen2}."
            )

    # Audit Checkpoint Champion Epochs
    print("\n" + "=" * 70)
    print("🔎 SAVED CHECKPOINTS CHAMPION EPOCHS AUDIT")
    print("=" * 70)

    for gen in [gen1, gen2]:
        ckpt_dir = Path(f"output/model_checkpoints/walk_forward_gen{gen}")
        if ckpt_dir.exists():
            ckpts = sorted(list(ckpt_dir.glob("*.pt")))
            print(f"\nGen {gen} Checkpoints ({len(ckpts)} files in {ckpt_dir.name}):")
            for cp in ckpts:
                try:
                    payload = torch.load(cp, map_location="cpu", weights_only=False)
                    ep = payload.get("epoch", "N/A")
                    fit = payload.get("val_fitness", "N/A")
                    sh = payload.get("val_sharpe", "N/A")
                    fit_str = f"{fit:.4f}" if isinstance(fit, float) else str(fit)
                    sh_str = f"{sh:.3f}" if isinstance(sh, float) else str(sh)
                    print(
                        f"  • {cp.name:32s} | Best Epoch: {ep:>2} | Val Fitness: {fit_str:>8} | Val Sharpe: {sh_str:>6}"
                    )
                except Exception as ex:
                    print(f"  • {cp.name:32s} | Error reading: {ex}")
        else:
            print(f"⚠️ Checkpoint directory not found: {ckpt_dir}")


if __name__ == "__main__":
    main()

🔍 DEEP DIFF: GENERATION 24 vs GENERATION 25 CONFIGURATIONS
✅ [trading_config] is 100% bitwise IDENTICAL between Gen 24 and Gen 25.
✅ [chunk_specs] is 100% bitwise IDENTICAL between Gen 24 and Gen 25.
✅ [data_lineage] is 100% bitwise IDENTICAL between Gen 24 and Gen 25.
✅ [environment] is 100% bitwise IDENTICAL between Gen 24 and Gen 25.

🔎 SAVED CHECKPOINTS CHAMPION EPOCHS AUDIT

Gen 24 Checkpoints (15 files in walk_forward_gen24):
  • model_chunk0_s101_champion.pt    | Best Epoch: 58 | Val Fitness:   0.0929 | Val Sharpe:  1.248
  • model_chunk0_s42_champion.pt     | Best Epoch: 88 | Val Fitness:   0.3933 | Val Sharpe:  1.588
  • model_chunk0_s777_champion.pt    | Best Epoch: 59 | Val Fitness:   0.0256 | Val Sharpe:  1.068
  • model_chunk1_s101_champion.pt    | Best Epoch:  6 | Val Fitness:   0.2370 | Val Sharpe:  0.491
  • model_chunk1_s42_champion.pt     | Best Epoch: 13 | Val Fitness:   0.0839 | Val Sharpe:  0.366
  • model_chunk1_s777_champion.pt    | Best Epoch:  9 | Val Fitness: 